In [ ]:
!nvidia-smi

Tue Sep 15 07:39:08 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   44C    P8             12W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
!pip install -q -U transformers accelerate bitsandbytes sentencepiece

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 98.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 34.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 20.6 MB/s eta 0:00:00


In [ ]:
from huggingface_hub import notebook_login

notebook_login()

In [ ]:
import os
import csv

from google.colab import files

os.makedirs("crossentropy", exist_ok=True)

uploaded = files.upload()

print()
for name in sorted(uploaded):
    with open(name, encoding="utf-8") as f:
        n = len(list(csv.DictReader(f, delimiter="\t")))
    print("  {}  ({} texts)".format(name, n))

TypeError: 'NoneType' object is not subscriptable

In [ ]:
MODELS = {
    "krikri":  "ilsp/Llama-Krikri-8B-Instruct",
    "meltemi": "ilsp/Meltemi-7B-Instruct-v1.5",
    "llama":   "meta-llama/Llama-3.1-8B-Instruct",
}

LOAD_IN_4BIT = True    # same as the inference run

INPUT_FILES = {
    "sentence":  "crossentropy_sentences.tsv",
    "paragraph": "crossentropy_paragraphs.tsv",
}

print("4-bit:", LOAD_IN_4BIT)

4-bit: True


In [ ]:
import gc
import json

import torch
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig


def load_model(hf_id):
    """Load one model and its tokenizer."""
    tokenizer = AutoTokenizer.from_pretrained(hf_id)

    settings = {"device_map": "auto", "torch_dtype": torch.float16}
    if LOAD_IN_4BIT:
        settings["quantization_config"] = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_quant_type="nf4",
        )

    model = AutoModelForCausalLM.from_pretrained(hf_id, **settings)
    model.eval()
    return model, tokenizer


def measure_surprise(model, tokenizer, text):
    """How surprised the model is by this text.

    Returns (average loss per token, number of tokens).
    Lower loss means the text was easier to predict.
    """
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    n_tokens = inputs["input_ids"].shape[1]

    # passing labels makes the model predict each token from the ones before it
    # and report how wrong it was. No text is generated.
    with torch.no_grad():
        outputs = model(**inputs, labels=inputs["input_ids"])

    return outputs.loss.item(), n_tokens


def read_texts(filename, level):
    """Read one input file into a list of rows."""
    texts = []

    with open(filename, encoding="utf-8") as f:
        for row in csv.DictReader(f, delimiter="\t"):
            texts.append({
                "text_id": row["text_id"],
                "level": level,
                "tasks": row["tasks"],
                "pharasiot_text": row["pharasiot_text"],
                "greek_text": row["greek_text"],
            })

    return texts


def run(model_name):
    """Measure surprise for every text, in both varieties, and save the results."""
    print("loading", model_name)
    model, tokenizer = load_model(MODELS[model_name])

    results = []

    for level, filename in INPUT_FILES.items():
        texts = read_texts(filename, level)

        for row in tqdm(texts, desc="{} {}".format(model_name, level)):
            for variety in ["pharasiot", "greek"]:
                text = row[variety + "_text"]
                loss, n_tokens = measure_surprise(model, tokenizer, text)

                results.append({
                    "text_id": row["text_id"],
                    "level": row["level"],
                    "tasks": row["tasks"],
                    "model": model_name,
                    "variety": variety,
                    "n_tokens": n_tokens,
                    "loss": round(loss, 4),
                })

    path = "crossentropy/{}.jsonl".format(model_name)
    with open(path, "w", encoding="utf-8") as f:
        for row in results:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")

    print("wrote {}  ({} rows)".format(path, len(results)))

    # free the GPU before the next model
    del model
    gc.collect()
    torch.cuda.empty_cache()

In [ ]:
run("krikri")

loading krikri


config.json:   0%|          | 0.00/955 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/55.5k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 19.5MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/752 [00:00<?, ?B/s]

... (5 outputs left out)

generation_config.json:   0%|          | 0.00/184 [00:00<?, ?B/s]

krikri sentence:   0%|          | 0/44 [00:00<?, ?it/s]

krikri paragraph:   0%|          | 0/15 [00:00<?, ?it/s]

wrote crossentropy/krikri.jsonl  (118 rows)


In [ ]:
run("meltemi")

loading meltemi


config.json:   0%|          | 0.00/713 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/3.65k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/3.97M [00:00<?, ?B/s]

tokenizer.model: reconstructing file:   0%|          |  0.00B / 1.18MB            

tokenizer.model: downloading bytes:           |  0.00B            

... (6 outputs left out)

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

meltemi sentence:   0%|          | 0/44 [00:00<?, ?it/s]

meltemi paragraph:   0%|          | 0/15 [00:00<?, ?it/s]

wrote crossentropy/meltemi.jsonl  (118 rows)


In [ ]:
run("llama")

loading llama


config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/55.4k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/23.9k [00:00<?, ?B/s]

... (3 outputs left out)

generation_config.json:   0%|          | 0.00/184 [00:00<?, ?B/s]

llama sentence:   0%|          | 0/44 [00:00<?, ?it/s]

llama paragraph:   0%|          | 0/15 [00:00<?, ?it/s]

wrote crossentropy/llama.jsonl  (118 rows)


In [ ]:
for name in sorted(os.listdir("crossentropy")):
    rows = [json.loads(line) for line in open("crossentropy/" + name, encoding="utf-8")]
    print(name, " n =", len(rows))

    for level in ["sentence", "paragraph"]:
        for variety in ["pharasiot", "greek"]:
            picked = [r for r in rows if r["level"] == level and r["variety"] == variety]
            if not picked:
                continue
            mean_loss = sum(r["loss"] for r in picked) / len(picked)
            mean_tokens = sum(r["n_tokens"] for r in picked) / len(picked)
            print("   {:10} {:10} mean loss {:.3f}   mean tokens {:.1f}".format(
                level, variety, mean_loss, mean_tokens))

    broken = [r for r in rows if r["loss"] <= 0 or r["loss"] != r["loss"]]
    print("   broken rows:", len(broken))
    print()

krikri.jsonl  n = 118
   sentence   pharasiot  mean loss 6.772   mean tokens 27.3
   sentence   greek      mean loss 4.412   mean tokens 20.1
   paragraph  pharasiot  mean loss 5.103   mean tokens 106.2
   paragraph  greek      mean loss 3.052   mean tokens 77.4
   broken rows: 0

llama.jsonl  n = 118
   sentence   pharasiot  mean loss 6.176   mean tokens 34.3
   sentence   greek      mean loss 3.695   mean tokens 28.7
   paragraph  pharasiot  mean loss 4.689   mean tokens 133.7
   paragraph  greek      mean loss 2.669   mean tokens 111.3
   broken rows: 0

meltemi.jsonl  n = 118
   sentence   pharasiot  mean loss 6.447   mean tokens 27.0
   sentence   greek      mean loss 4.078   mean tokens 18.8
   paragraph  pharasiot  mean loss 5.096   mean tokens 105.6
   paragraph  greek      mean loss 3.025   mean tokens 73.3
   broken rows: 0



In [ ]:
!zip -qr crossentropy_results.zip crossentropy

from google.colab import files

files.download("crossentropy_results.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>